## Data Preprocessing

In [2]:
# Import necessary libraries
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download required NLTK data assets (run once in Colab)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

df = pd.read_csv('/content/knowledge_base_improved 1.csv')

# 2. Convert keywords list into a space-separated string
df['keywords_str'] = df['keywords'].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x))

# 3. Combine 'title', 'content', and 'keywords' into a unified text representation
df['combined_text'] = (
    df['title'].fillna('') + " " +
    df['content'].fillna('') + " " +
    df['keywords_str'].fillna('')
)

# 4. Initialize Stopwords and Lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """
    Cleans, lowercases, tokenizes, removes stopwords, and lemmatizes input text.
    """
    if not isinstance(text, str):
        return []

    # Step A: Lowercase
    text = text.lower()

    # Step B: Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)

    # Step C: Tokenize
    tokens = word_tokenize(text)

    # Step D: Remove stopwords and short tokens, then apply lemmatization
    filtered_tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words and len(word) > 1
    ]

    return filtered_tokens

# 5. Apply preprocessing to the combined text column
df['cleaned_tokens'] = df['combined_text'].apply(preprocess_text)

# Create a space-joined string version for TF-IDF
df['cleaned_text_str'] = df['cleaned_tokens'].apply(lambda tokens: ' '.join(tokens))

# Display the result
print("--- Processed Data with Keywords and Lemmatization ---")
print(df[['document_id', 'combined_text', 'cleaned_tokens']].head())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


--- Processed Data with Keywords and Lemmatization ---
  document_id                                      combined_text  \
0    KB000001  Unable to Access Your Login Password Step by S...   
1    KB000002  Login Troubleshooting the Easy Way Problems wi...   
2    KB000003  Setting Up Extra Login Step You can set up you...   
3    KB000004  Verification Code Troubleshooting Explained Mo...   
4    KB000005  How to Set Up Your Authentication App the Easy...   

                                      cleaned_tokens  
0  [unable, access, login, password, step, step, ...  
1  [login, troubleshooting, easy, way, problem, l...  
2  [setting, extra, login, step, set, twostep, ve...  
3  [verification, code, troubleshooting, explaine...  
4  [set, authentication, app, easy, way, setting,...  


## Model Building || Model Training

### Word2Vec model

In [3]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 78.7 MB/s eta 0:00:00


In [4]:
# Import required libraries for Word2Vec training
from gensim.models import Word2Vec
import numpy as np

# 1. Prepare sentences/tokens from our preprocessed dataset
# Assuming 'cleaned_tokens' is the column containing lists of tokens from our previous step
sentences = df['cleaned_tokens'].tolist() # creating corpus

print(f"Total training documents/sentences: {len(sentences)}")

# 2. Initialize and Train the Word2Vec Model
# vector_size: dimensionality of word vectors (e.g., 100 dimensions)
# window: maximum distance between the current and predicted word within a sentence
# min_count: ignores words with total frequency lower than this
# workers: number of CPU cores to use for training
# sg: training algorithm: 1 for skip-gram; otherwise CBOW (0)
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1
)

print("--- Word2Vec Model Trained Successfully ---")
print(f"Vocabulary size: {len(w2v_model.wv)}")

# 3. Define a function to generate document vectors by averaging word embeddings
def get_document_vector(tokens, model):
    """
    Computes the mean vector of all words in a document that exist in the Word2Vec vocabulary.
    """
    # Filter tokens that are present in the Word2Vec vocabulary
    valid_tokens = [token for token in tokens if token in model.wv]

    if not valid_tokens:
        # Return a zero vector if no tokens are found in vocabulary
        return np.zeros(model.vector_size)

    # Calculate the mean of the word vectors
    doc_vector = np.mean([model.wv[token] for token in valid_tokens], axis=0)
    return doc_vector

# 4. Apply the function to create a document embedding column
df['w2v_doc_vector'] = df['cleaned_tokens'].apply(lambda tokens: get_document_vector(tokens, w2v_model))

# Display sample vector shape
print(f"Sample Document Vector Shape: {df['w2v_doc_vector'].iloc[0].shape}")

Total training documents/sentences: 50000
--- Word2Vec Model Trained Successfully ---
Vocabulary size: 697
Sample Document Vector Shape: (100,)


### FastText model

In [5]:
# Import required libraries for FastText training
from gensim.models import FastText
import numpy as np

# 1. Prepare sentences/tokens from our preprocessed dataset
# Assuming 'cleaned_tokens' is already available from your data preparation step
sentences = df['cleaned_tokens'].tolist()

print(f"Total training documents/sentences for FastText: {len(sentences)}")

# 2. Initialize and Train the FastText Model
# vector_size: dimensionality of word vectors (e.g., 100 dimensions)
# window: maximum distance between current and predicted word
# min_count: minimum word frequency threshold
# min_n / max_n: character n-gram lengths for sub-word representation
# workers: number of CPU cores
ft_model = FastText(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    min_n=3,
    max_n=6,
    workers=4,
    sg=1
)

print("--- FastText Model Trained Successfully ---")
print(f"FastText Vocabulary size: {len(ft_model.wv)}")

# 3. Define a fallback function for document vectorization
def get_fasttext_document_vector(tokens, model):
    """
    Computes the mean vector of all words in a document using FastText.
    Because FastText uses character n-grams, even unseen words will
    return a valid non-zero vector instead of failing.
    """
    if not tokens:
        return np.zeros(model.vector_size)

    # FastText generates vectors for any string via sub-words,
    # but averaging tokens gives us the overall document representation.
    doc_vector = np.mean([model.wv[token] for token in tokens], axis=0)
    return doc_vector

# 4. Apply the function to create a FastText document embedding column
df['ft_doc_vector'] = df['cleaned_tokens'].apply(lambda tokens: get_fasttext_document_vector(tokens, ft_model))

# 5. Demonstration of handling an unseen/out-of-vocabulary word
unseen_word = "unseenkeyword"
print(f"\n--- Testing Unseen Word Handling ---")
print(f"Is '{unseen_word}' in Word2Vec? {unseen_word in w2v_model.wv}")
print(f"Is '{unseen_word}' in FastText? {unseen_word in ft_model.wv}") # Might be False, but FastText can still vectorize it!
print(f"FastText vector for unseen word shape: {ft_model.wv[unseen_word].shape}")

Total training documents/sentences for FastText: 50000
--- FastText Model Trained Successfully ---
FastText Vocabulary size: 697

--- Testing Unseen Word Handling ---
Is 'unseenkeyword' in Word2Vec? False
Is 'unseenkeyword' in FastText? True
FastText vector for unseen word shape: (100,)


### TF-IDF model

In [6]:
# Import required libraries for TF-IDF vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Use the space-joined preprocessed text strings from our data preparation step
# Assuming 'cleaned_text_str' is our dataframe column containing the processed text
corpus = df['cleaned_text_str'].tolist()

print(f"Total documents in TF-IDF corpus: {len(corpus)}")

# 2. Initialize and Fit the TfidfVectorizer
# max_features: limits the vocabulary size to the top N most frequent terms (optional, can be None)
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Learn the vocabulary and compute TF-IDF sparse matrix for the document collection
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

print("--- TF-IDF Model Trained Successfully ---")
print(f"TF-IDF Matrix Shape (Documents x Vocabulary): {tfidf_matrix.shape}")
print(f"TF-IDF Vocabulary Size: {len(tfidf_vectorizer.get_feature_names_out())}")

Total documents in TF-IDF corpus: 50000
--- TF-IDF Model Trained Successfully ---
TF-IDF Matrix Shape (Documents x Vocabulary): (50000, 697)
TF-IDF Vocabulary Size: 697


## Model Testing

### Word2Vec testing

In [9]:
# Import required libraries for similarity calculation
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def search_word2vec(query_text, df, model, top_k=3):
    """
    Takes a user query, preprocesses it, computes its Word2Vec embedding,
    and ranks documents using cosine similarity.
    """
    # 1. Preprocess the query using our established preprocess_text function
    query_tokens = preprocess_text(query_text)

    # 2. Convert query tokens into an averaged vector
    valid_tokens = [token for token in query_tokens if token in model.wv]
    if not valid_tokens:
        query_vector = np.zeros(model.vector_size)
    else:
        query_vector = np.mean([model.wv[token] for token in valid_tokens], axis=0)

    # Reshape for scikit-learn cosine_similarity (requires 2D array: shape (1, vector_size))
    query_vector = query_vector.reshape(1, -1)

    # 3. Stack all pre-calculated document vectors into a 2D matrix
    doc_vectors = np.vstack(df['w2v_doc_vector'].values)

    # 4. Compute cosine similarity between the query and all document vectors
    similarities = cosine_similarity(query_vector, doc_vectors)[0]

    # 5. Attach scores to the dataframe and rank them (highest similarity first)
    results_df = df.copy()
    results_df['similarity_score'] = similarities
    ranked_results = results_df.sort_values(by='similarity_score', ascending=False)

    # Return top-k documents based on user requirement 4.7
    return ranked_results[['document_id', 'title', 'similarity_score']].head(top_k)

# --- Example Execution ---
test_query = "MY app not working"
top_results = search_word2vec(test_query, df, w2v_model, top_k=5)

print(f"--- Search Results for Query: '{test_query}' (Word2Vec) ---")
print(top_results.to_string(index=False))

--- Search Results for Query: 'MY app not working' (Word2Vec) ---
document_id                                        title  similarity_score
   KB045025          App Crash Not Responding on the Web          0.741128
   KB025324 What to Do When App Closing Fails in the App          0.738222
   KB026584                  App Closing Stopped Working          0.736651
   KB025276          Solving App Crash Issues in Minutes          0.732443
   KB025744        App Freezing Not Responding on Mobile          0.732256


### FastText testing

In [12]:
# Import required libraries for similarity calculation
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def search_fasttext(query_text, df, model, top_k=3):
    """
    Takes a user query, preprocesses it, computes its FastText embedding
    (handling out-of-vocabulary words via sub-words), and ranks documents
    using cosine similarity.
    """
    # 1. Preprocess the query using our established preprocess_text function
    query_tokens = preprocess_text(query_text)

    # 2. Convert query tokens into an averaged vector using FastText
    if not query_tokens:
        query_vector = np.zeros(model.vector_size)
    else:
        # FastText can generate vectors for any token using character n-grams
        query_vector = np.mean([model.wv[token] for token in query_tokens], axis=0)

    # Reshape for scikit-learn cosine_similarity (requires 2D array: shape (1, vector_size))
    query_vector = query_vector.reshape(1, -1)

    # 3. Stack all pre-calculated FastText document vectors into a 2D matrix
    doc_vectors = np.vstack(df['ft_doc_vector'].values)

    # 4. Compute cosine similarity between the query and all document vectors
    similarities = cosine_similarity(query_vector, doc_vectors)[0]

    # 5. Attach scores to the dataframe and rank them (highest similarity first)
    results_df = df.copy()
    results_df['similarity_score'] = similarities
    ranked_results = results_df.sort_values(by='similarity_score', ascending=False)

    # Return top-k documents based on user requirement 4.7
    return ranked_results[['document_id', 'title', 'similarity_score']].head(top_k)

# --- Example Execution (Testing with an unseen term / typo scenario) ---
test_query = "MY app not worng"
top_results_ft = search_fasttext(test_query, df, ft_model, top_k=5)

print(f"--- Search Results for Query: '{test_query}' (FastText) ---")
print(top_results_ft.to_string(index=False))

--- Search Results for Query: 'MY app not worng' (FastText) ---
document_id                                             title  similarity_score
   KB024664           Resolving App Crash Issues Step by Step          0.752869
   KB025756               App Closing Keeps Failing on Mobile          0.751795
   KB026152                  Fixing App Crash Problems on iOS          0.751672
   KB044953                      Fixing App Freezing Problems          0.750783
   KB045829 App Closing Troubleshooting for Business Accounts          0.750210


### TF-IDF testing

In [14]:
# Import required libraries for similarity calculation
from sklearn.metrics.pairwise import cosine_similarity

def search_tfidf(query_text, df, vectorizer, tfidf_matrix, top_k=3):
    """
    Takes a user query, preprocesses it into a text string, transforms it
    using the fitted TF-IDF vectorizer, and ranks documents using cosine similarity.
    """
    # 1. Preprocess the query and join back into a string for TF-IDF
    query_tokens = preprocess_text(query_text)
    query_text_str = " ".join(query_tokens)

    # 2. Transform the query text into a TF-IDF sparse vector using the fitted vectorizer
    query_tfidf_vector = vectorizer.transform([query_text_str])

    # 3. Compute cosine similarity between the query TF-IDF vector and all document vectors
    similarities = cosine_similarity(query_tfidf_vector, tfidf_matrix)[0]

    # 4. Attach scores to the dataframe and rank them (highest similarity first)
    results_df = df.copy()
    results_df['similarity_score'] = similarities
    ranked_results = results_df.sort_values(by='similarity_score', ascending=False)

    # Return top-k documents based on user requirement 4.7
    return ranked_results[['document_id', 'title', 'similarity_score']].head(top_k)

# --- Example Execution ---
test_query = "MY app not working"
top_results_tfidf = search_tfidf(test_query, df, tfidf_vectorizer, tfidf_matrix, top_k=5)

print(f"--- Search Results for Query: '{test_query}' (TF-IDF Baseline) ---")
print(top_results_tfidf.to_string(index=False))

--- Search Results for Query: 'MY app not working' (TF-IDF Baseline) ---
document_id                                            title  similarity_score
   KB045157          App Freezing Stopped Working in the App          0.346132
   KB027298               App Update Problem Stopped Working          0.340504
   KB046380 Why Your Failed Update Is Not Working in the App          0.340167
   KB046435                     Stuck Update Stopped Working          0.333369
   KB028029           App Sign-In Stopped Working on the Web          0.332756


### Hybrid : Word2Vec + FastText(fallback)

In [16]:
# Import required libraries for similarity calculation
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def search_hybrid_w2v_ft(query_text, df, w2v_model, ft_model, top_k=3):
    """
    Takes a user query and builds a hybrid embedding:
    - Primary: Uses Word2Vec vector if the token exists in its vocabulary.
    - Fallback: Uses FastText sub-word vector if the token is out-of-vocabulary (OOV) for Word2Vec.
    Then ranks documents using cosine similarity.
    """
    # 1. Preprocess the query using our established function
    query_tokens = preprocess_text(query_text)

    if not query_tokens:
        print("Warning: Query resulted in no valid tokens after preprocessing.")
        return df[['document_id', 'title']].head(0)

    token_vectors = []
    for token in query_tokens:
        if token in w2v_model.wv:
            # Primary Model: Word2Vec handles known words
            token_vectors.append(w2v_model.wv[token])
        else:
            # Fallback Model: FastText takes over for unseen/OOV words via sub-words
            token_vectors.append(ft_model.wv[token])
            print(f"-> [Fallback Triggered] Unseen word '{token}' not in Word2Vec. FastText took over.")

    # 2. Average the token vectors to create the final query embedding
    query_vector = np.mean(token_vectors, axis=0).reshape(1, -1)

    # 3. Stack pre-calculated Word2Vec document vectors for comparison space
    doc_vectors = np.vstack(df['w2v_doc_vector'].values)

    # 4. Compute cosine similarity between the hybrid query vector and documents
    similarities = cosine_similarity(query_vector, doc_vectors)[0]

    # 5. Attach scores to the dataframe and rank them (highest similarity first)
    results_df = df.copy()
    results_df['similarity_score'] = similarities
    ranked_results = results_df.sort_values(by='similarity_score', ascending=False)

    # Return top-k documents based on user requirement 4.7
    return ranked_results[['document_id', 'title', 'similarity_score']].head(top_k)

# --- Example Execution (Testing with an unseen word/typo) ---
test_query = "MY app not worng"
top_results_hybrid = search_hybrid_w2v_ft(test_query, df, w2v_model, ft_model, top_k=5)

print(f"\n--- Search Results for Query: '{test_query}' (Hybrid Pipeline) ---")
print(top_results_hybrid.to_string(index=False))

-> [Fallback Triggered] Unseen word 'worng' not in Word2Vec. FastText took over.

--- Search Results for Query: 'MY app not worng' (Hybrid Pipeline) ---
document_id                                            title  similarity_score
   KB045680 A Guide to Setting Up Fresh Install the Easy Way          0.520325
   KB024719   Set Up App Reinstall Step by Step Step by Step          0.518490
   KB046131      A Guide to Setting Up Mobile App on the Web          0.517128
   KB026123  A Guide to Setting Up App Reinstall Quick Guide          0.516350
   KB025019              Setting Up Fresh Install on Android          0.515160


## Model Evaluation

In [21]:
# Import required libraries for formatting and evaluation
import pandas as pd

def evaluate_and_compare_models(query_text, df, w2v_model, ft_model, tfidf_vectorizer, tfidf_matrix, top_k=3):
    """
    Executes search across TF-IDF, Word2Vec, and the Hybrid Fallback Pipeline,
    then merges the results into a comparative evaluation table.
    """
    print(f"==================================================")
    print(f" EVALUATION COMPARISON FOR QUERY: '{query_text}'")
    print(f"==================================================")

    # 1. Get results from TF-IDF Baseline
    tfidf_res = search_tfidf(query_text, df, tfidf_vectorizer, tfidf_matrix, top_k=top_k).reset_index(drop=True)
    tfidf_res.rename(columns={'title': 'TF-IDF Title', 'similarity_score': 'TF-IDF Score'}, inplace=True)

    # 2. Get results from Word2Vec Model
    w2v_res = search_word2vec(query_text, df, w2v_model, top_k=top_k).reset_index(drop=True)
    w2v_res.rename(columns={'title': 'Word2Vec Title', 'similarity_score': 'Word2Vec Score'}, inplace=True)

    # 3. Get results from FastText Pipeline
    ft_res = search_fasttext(query_text, df, ft_model, top_k=top_k).reset_index(drop=True)
    ft_res.rename(columns={'title': 'FastText Title', 'similarity_score': 'FastText Score'}, inplace=True)

    # 4. Combine into a side-by-side comparison table
    comparison_table = pd.concat([
        tfidf_res[['document_id', 'TF-IDF Title', 'TF-IDF Score']],
        w2v_res[['Word2Vec Title', 'Word2Vec Score']],
        ft_res[['FastText Title', 'FastText Score']]
    ], axis=1)

    return comparison_table

# --- Example Execution ---
evaluation_query = "My accunt hacked my someone how fix this"
comparison_df = evaluate_and_compare_models(evaluation_query, df, w2v_model, ft_model, tfidf_vectorizer, tfidf_matrix, top_k=4)

# Display the final structured comparison table
print("\n--- Model Evaluation Comparison Table ---")
print(comparison_df.to_string(index=False))

 EVALUATION COMPARISON FOR QUERY: 'My accunt hacked my someone how fix this'

--- Model Evaluation Comparison Table ---
document_id                                             TF-IDF Title  TF-IDF Score                               Word2Vec Title  Word2Vec Score                                     FastText Title  FastText Score
   KB043814            Fix Hacked Account Access Problems in Minutes      0.707133     Fix Closed Ticket Access Problems on iOS        0.723458 Fix Compromised Account Access Problems in Minutes        0.748304
   KB020647 Fix Hacked Account Access Problems for Business Accounts      0.702272 Regain Access to Your Old Ticket Quick Guide        0.706452     Fix Stolen Account Access Problems Quick Guide        0.745994
   KB020635                   Recover Your Hacked Account on the Web      0.698470 Unable to Access Your Old Ticket Quick Guide        0.705523    Resetting a Forgotten Hacked Account in Minutes        0.744558
   KB022399           Resetting a Fo

### When the user query goes out of vocabulary, Word2Vec fails but FastText performed well, TF-IDF seems like works well but it actually match based on keywords. Based on the similarity score FASTTEXT perform well.

### So, we will combine Word2Vec + FastText(fallback model) to fetch documents, we ignore TF-IDF here because we need to retrive the documents by understanding the semantic meaning of user's query not on keyword matching

## Model deployment

In [22]:
# Import required serialization and Colab utilities
import pickle
import joblib
from google.colab import files

print("--- Starting Model Export Process ---")

# 1. Save Gensim Word2Vec and FastText models using their native .save() methods
w2v_filename = "word2vec_model.model"
ft_filename = "fasttext_model.model"

w2v_model.save(w2v_filename)
ft_model.save(ft_filename)
print(f"-> Saved Word2Vec to: {w2v_filename}")
print(f"-> Saved FastText to: {ft_filename}")

# 2. Save TF-IDF Vectorizer and Matrix using joblib/pickle
tfidf_vectorizer_filename = "tfidf_vectorizer.pkl"
tfidf_matrix_filename = "tfidf_matrix.pkl"

joblib.dump(tfidf_vectorizer, tfidf_vectorizer_filename)
joblib.dump(tfidf_matrix, tfidf_matrix_filename)
print(f"-> Saved TF-IDF Vectorizer to: {tfidf_vectorizer_filename}")
print(f"-> Saved TF-IDF Matrix to: {tfidf_matrix_filename}")

# 3. Automatically trigger downloads to your local machine via Google Colab
files.download(w2v_filename)
files.download(ft_filename)
files.download(tfidf_vectorizer_filename)
files.download(tfidf_matrix_filename)

print("--- All Models Exported and Downloaded Successfully! ---")

--- Starting Model Export Process ---
-> Saved Word2Vec to: word2vec_model.model
-> Saved FastText to: fasttext_model.model
-> Saved TF-IDF Vectorizer to: tfidf_vectorizer.pkl
-> Saved TF-IDF Matrix to: tfidf_matrix.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

--- All Models Exported and Downloaded Successfully! ---


In [26]:
import joblib

# Save the entire dataframe containing your precomputed document vectors
df.to_pickle("processed_documents_df.pkl")
print("-> Saved Document Vectors DataFrame to: processed_documents_df.pkl")

# Trigger download to your local machine
from google.colab import files
files.download("processed_documents_df.pkl")

-> Saved Document Vectors DataFrame to: processed_documents_df.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
# Import required libraries for loading and testing
from gensim.models import Word2Vec, FastText
import joblib
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("--- Loading Saved Models from Disk ---")

# 1. Reload Gensim Word2Vec and FastText models using their native .load() methods
loaded_w2v_model = Word2Vec.load("word2vec_model.model")
loaded_ft_model = FastText.load("fasttext_model.model")

print(f"-> Word2Vec reloaded successfully. Vocabulary size: {len(loaded_w2v_model.wv)}")
print(f"-> FastText reloaded successfully. Vocabulary size: {len(loaded_ft_model.wv)}")

# 2. Reload TF-IDF Vectorizer and Matrix using joblib
loaded_tfidf_vectorizer = joblib.load("tfidf_vectorizer.pkl")
loaded_tfidf_matrix = joblib.load("tfidf_matrix.pkl")

print(f"-> TF-IDF Vectorizer reloaded successfully.")
print(f"-> TF-IDF Matrix reloaded successfully. Shape: {loaded_tfidf_matrix.shape}")

# 3. Quick Sanity Check: Test a search query using the reloaded models
def test_reloaded_hybrid_search(query_text, top_k=2):
    """
    Runs a quick search using the newly reloaded models to ensure full functionality.
    """
    query_tokens = preprocess_text(query_text)

    if not query_tokens:
        print("Warning: Query resulted in no valid tokens.")
        return

    token_vectors = []
    for token in query_tokens:
        if token in loaded_w2v_model.wv:
            token_vectors.append(loaded_w2v_model.wv[token])
        else:
            token_vectors.append(loaded_ft_model.wv[token])

    query_vector = np.mean(token_vectors, axis=0).reshape(1, -1)

    # Using our pre-computed document vectors from df for this test
    doc_vectors = np.vstack(df['w2v_doc_vector'].values)
    similarities = cosine_similarity(query_vector, doc_vectors)[0]

    # Print top results
    print(f"\n--- Sanity Check Search for: '{query_text}' ---")
    print(f"Top Similarity Score: {np.max(similarities):.4f}")

# Execute sanity check
test_reloaded_hybrid_search("Hacked account")
print("\n--- All Models Verified and Ready for Web App Integration! ---")

--- Loading Saved Models from Disk ---
-> Word2Vec reloaded successfully. Vocabulary size: 697
-> FastText reloaded successfully. Vocabulary size: 697
-> TF-IDF Vectorizer reloaded successfully.
-> TF-IDF Matrix reloaded successfully. Shape: (50000, 697)

--- Sanity Check Search for: 'Hacked account' ---
Top Similarity Score: 0.7971

--- All Models Verified and Ready for Web App Integration! ---


In [25]:
df.columns

Index(['document_id', 'category', 'title', 'content', 'keywords',
       'keywords_str', 'combined_text', 'cleaned_tokens', 'cleaned_text_str',
       'w2v_doc_vector', 'ft_doc_vector'],
      dtype='object')

## END !!!